# 임포트 및 초기 설정

In [ ]:
import os, sys, random, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('.')
from src.utils import set_seed, make_run_dir
from src.visualization import (plot_loss_history, plot_weight_tradeoff,
                                plot_confusion_matrix, plot_model_comparison_pareto,
                                plot_model_comparison_bar,  plot_gamma_sweep,
                                plot_weight_sweep, show_gradcam_grid)
from src.engine import measure_inference_speed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0  # Windows/Jupyter에서는 0이 가장 안정적
print(f"Device: {device} | num_workers: {NUM_WORKERS}")

run_dirs = make_run_dir('./outputs')

# 커스텀 Dataset 정의

In [ ]:
# ── 커스텀 Dataset (인덱스 기반, fold 분할에 필요)
class ScrewDataset(Dataset):
    def __init__(self, img_paths, labels, transform=None):
        self.img_paths = img_paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]
        

# ── Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Dataset, FocalLoss, Transform 정의 완료")

# 데이터 로딩

In [ ]:
DATA_DIR = './data/k-fold_data'
CLASS_NAMES = ['good', 'type1', 'type2', 'type3', 'type4', 'type5']

# 이진 라벨로 통일: good=0, type1~type5=1(bad)
all_paths, all_labels = [], []

valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')

for class_name in CLASS_NAMES:
    class_dir = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_dir):
        raise FileNotFoundError(f"클래스 폴더 없음: {class_dir}")

    binary_label = 0 if class_name == 'good' else 1

    for fname in sorted(os.listdir(class_dir)):
        if fname.lower().endswith(valid_exts):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(binary_label)

all_labels_np = np.array(all_labels, dtype=np.int64)

# 분포 확인
print(f"전체 샘플: {len(all_paths)}장")
print(f"  good(0): {(all_labels_np == 0).sum()}장")
print(f"  bad (1): {(all_labels_np == 1).sum()}장")

if len(all_paths) == 0:
    raise RuntimeError("로드된 이미지가 0장입니다. 확장자/경로를 확인하세요.")

# 모델 준비

In [ ]:
def build_model_bin(model_name):
    """이진 분류(num_classes=2)로 헤드 교체"""
    if model_name == 'resnet18':
        m = models.resnet18(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, 2)
    elif model_name == 'mobilenet_v2':
        m = models.mobilenet_v2(weights='DEFAULT')
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    elif model_name == 'vgg16':
        m = models.vgg16(weights='DEFAULT')
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, 2)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return m


def f2_score(precision, recall):
    denom = 4 * precision + recall
    return 5 * precision * recall / denom if denom > 0 else 0.0

# 모델 학습

In [ ]:
def run_kfold_experiment(model_name, all_paths, all_labels_np,
                         bad_weight_list, n_splits=5,
                         num_epochs=50, batch_size=16,
                         num_workers=NUM_WORKERS,
                         device=device, run_dirs=run_dirs,
                         save_best_models=True,
                         early_stop_patience=10):
    """Stratified 5-fold + Weighted CrossEntropy(bad weight sweep).

    - labels: good=0, bad=1
    - criterion: CrossEntropyLoss(weight=[1.0, bad_weight])
    """

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    bad_weight_list = list(bad_weight_list)

    # bad_weight별 fold F2 저장
    weight_f2_map = {w: [] for w in bad_weight_list}
    weight_predictions = {w: {'true': [], 'pred': []} for w in bad_weight_list}
    loss_histories = {w: {} for w in bad_weight_list}
    fold_scores = {w: {} for w in bad_weight_list}

    # fold/weight best 모델 저장 경로
    weights_dir = run_dirs.get('weights', './outputs')
    kfold_dir = os.path.join(weights_dir, 'kfold_weighted_ce')
    if save_best_models:
        os.makedirs(kfold_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"모델: {model_name.upper()} | {n_splits}-Fold CV | bad_weight sweep: {bad_weight_list}")
    print(f"{'='*60}")

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels_np)):
        print(f"\n[Fold {fold_idx+1}/{n_splits}]")

        train_paths_fold = [all_paths[i] for i in train_idx]
        val_paths_fold = [all_paths[i] for i in val_idx]

        train_labels_fold = all_labels_np[train_idx]

        train_ds = ScrewDataset(train_paths_fold, train_labels_fold, transform=train_transform)
        val_ds = ScrewDataset(val_paths_fold, all_labels_np[val_idx], transform=val_transform)

        class_counts = np.bincount(train_labels_fold, minlength=2)
        class_weight_vals = np.array([1.0, len(train_labels_fold) / class_counts[1]], dtype=np.float32)
        sample_w = torch.tensor(class_weight_vals[train_labels_fold], dtype=torch.float32)
        sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                                  num_workers=num_workers, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                                num_workers=num_workers, pin_memory=True)

        for bad_w in bad_weight_list:
            set_seed(42)
            model = build_model_bin(model_name).to(device)

            weights_tensor = torch.tensor([1.0, float(bad_w)], dtype=torch.float32, device=device)
            criterion = nn.CrossEntropyLoss(weight=weights_tensor)
            optimizer = optim.Adam(model.parameters(), lr=1e-4)
            scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

            best_val_loss = float('inf')
            best_state = None
            best_epoch = 0
            no_improve = 0
            train_loss_history = []
            val_loss_history = []

            for epoch in range(num_epochs):
                model.train()
                running_train_loss = 0.0

                for imgs, lbls in train_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(imgs), lbls)
                    loss.backward()
                    optimizer.step()
                    running_train_loss += loss.item()
                scheduler.step()

                avg_train_loss = running_train_loss / len(train_loader)
                train_loss_history.append(avg_train_loss)

                # Val loss (early stopping)
                model.eval()
                running_val_loss = 0.0
                with torch.no_grad():
                    for imgs, lbls in val_loader:
                        imgs, lbls = imgs.to(device), lbls.to(device)
                        running_val_loss += criterion(model(imgs), lbls).item()

                avg_val_loss = running_val_loss / len(val_loader)
                val_loss_history.append(avg_val_loss)

                if (epoch + 1) % 10 == 0 or epoch == 0:
                    print(f"  Epoch {epoch+1:02d} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")

                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    best_state = copy.deepcopy(model.state_dict())
                    best_epoch = epoch + 1
                    no_improve = 0
                else:
                    no_improve += 1

                if early_stop_patience > 0 and no_improve >= early_stop_patience:
                    print(f"    -> early stop at epoch {epoch+1}")
                    break

            if best_state is not None:
                model.load_state_dict(best_state)

            if save_best_models:
                save_path = os.path.join(kfold_dir, f"{model_name}_fold{fold_idx+1}_w{bad_w}.pth")
                torch.save(model.state_dict(), save_path)

            # Val 평가: bad(1) 클래스 F2
            model.eval()
            y_true, y_pred = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    preds = model(imgs.to(device)).argmax(dim=1).cpu().numpy()
                    y_true.extend(lbls.numpy())
                    y_pred.extend(preds)
            
            weight_predictions[bad_w]['true'].extend(y_true)
            weight_predictions[bad_w]['pred'].extend(y_pred)

            report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
            p_bad = report.get('1', {}).get('precision', 0.0)
            r_bad = report.get('1', {}).get('recall', 0.0)
            f2_bad = f2_score(p_bad, r_bad)

            weight_f2_map[bad_w].append(f2_bad)

            
            fold_scores[bad_w][fold_idx] = f2_bad

            loss_histories[bad_w][fold_idx] = {
                'train': train_loss_history,
                'val': val_loss_history
            }

            print(f"  w={bad_w} | F2(bad): {f2_bad:.4f} | best_epoch={best_epoch}\n")

    print(f"\n{'='*60}")
    print(f"{model_name.upper()} bad_weight sweep 결과 (5-fold 평균 F2(bad))")

    best_w, best_avg_f2 = None, -1.0

    for w, scores in weight_f2_map.items():
        avg = float(np.mean(scores))
        std = float(np.std(scores))
        print(f"  w={w} → 평균 F2: {avg:.4f} ± {std:.4f}")
        if avg > best_avg_f2:
            best_avg_f2, best_w = avg, w

    scores_list   = [fold_scores[best_w][i] for i in range(n_splits)]
    best_fold_idx = int(np.argmax(scores_list))
    best_fold_f2 = float(max(scores_list))

    print(f"\n최적 bad weight: {best_w}  (평균 F2: {best_avg_f2:.4f})")
    print(f' -> 해당 가중치 내 최고 성능 Fold : Fold {best_fold_idx + 1} (F2: {best_fold_f2:.4f})')
    print(f"{'='*60}")

    best_train_loss = loss_histories[best_w][best_fold_idx]['train']
    best_val_loss = loss_histories[best_w][best_fold_idx]['val']

    final_y_true = weight_predictions[best_w]['true']
    final_y_pred = weight_predictions[best_w]['pred']

    assert len(final_y_true) == len(all_paths), f"OOF 샘플 수 오류: {len(final_y_true)}장 (기대값: {len(all_paths)}장)"
    print(f"OOF 샘플 수 확인: {len(final_y_true)}장")

    return best_w, best_avg_f2, weight_f2_map, best_train_loss, best_val_loss, final_y_true, final_y_pred

# Weighted Sweep 준비

In [ ]:
BAD_WEIGHT_LIST = np.arange(1.0, 10.5, 0.5).tolist()  # 1.0 ~ 10.
BAD_WEIGHT_LIST = [1.0] # 테스트

In [ ]:
best_w_resnet, best_f2_resnet, w_map_resnet, resnet_best_train_loss, resnet_best_val_loss, resnet_final_y_true, resnet_final_y_pred = run_kfold_experiment(
    model_name='resnet18',
    all_paths=all_paths,
    all_labels_np=all_labels_np,
    bad_weight_list=BAD_WEIGHT_LIST,
    n_splits=5,
    num_epochs=40,
    batch_size=16,
    save_best_models=True,
)

print(f"\n[RESNET18] best bad weight = {best_w_resnet} | mean F2(bad) = {best_f2_resnet:.4f}")

In [ ]:
plot_weight_sweep(w_map_resnet, 'resnet18', run_dirs['figures'])

plot_loss_history(resnet_best_train_loss, 
                  resnet_best_val_loss, 
                  run_dirs['figures'], 
                  model_name='resnet18')

plot_confusion_matrix(resnet_final_y_true, 
                      resnet_final_y_pred, 
                      CLASS_NAMES, 
                      run_dirs['figures'], 
                      model_name='resnet18')

resnet_model = build_model_bin('resnet18').to(device)
best_model_path = os.path.join(run_dirs['weights'], 'resnet18_fold1_w1.0.pth')
resnet_model.load_state_dict(torch.load(best_model_path))

show_gradcam_grid(resnet_model, 
                  val_transform, 
                  'test', 
                  device, 
                  run_dirs['figures'], 
                  model_name='resnet18')